# Deploy OpenWeightAI to Cloud Run with an L4 GPU

This notebook builds the FastAPI gateway from GitHub and deploys it to Cloud Run next to a vLLM sidecar running `google/gemma-4-E2B-it` on one NVIDIA L4 GPU.

The model weights are downloaded by the vLLM sidecar at startup using your Hugging Face read token. They are not stored in GitHub.

## Before You Run

You need:

- A Google Cloud project with billing enabled.
- Permission to enable APIs, create Artifact Registry repositories, build containers, create service accounts, and deploy Cloud Run services.
- Hugging Face access to `google/gemma-4-E2B-it`: open `https://huggingface.co/google/gemma-4-E2B-it`, accept the terms if prompted, then create a read token at `https://huggingface.co/settings/tokens`.

Keep `MAX_INSTANCES = 1` for early testing so a mistake cannot fan out multiple GPU instances.

In [ ]:
# ---- Edit these values ----
PROJECT_ID = "YOUR_GOOGLE_CLOUD_PROJECT_ID"
REGION = "us-central1"  # L4 regions include us-central1, us-east4, europe-west1, europe-west4, asia-southeast1
SERVICE_NAME = "openweightai"
ARTIFACT_REPO = "openweightai"
GITHUB_REPO_URL = "https://github.com/jjtseng-jjtseng/openweightai.git"
MODEL_ID = "google/gemma-4-E2B-it"

# Cost guardrails. Increase later only after load testing.
MAX_INSTANCES = "1"
CONCURRENCY = 8

# True is easiest for a Raspberry Pi client: Cloud Run is public, but the app still requires X-API-Key.
# False keeps Cloud Run IAM private; your client then also needs a Google identity token.
ALLOW_PUBLIC_WITH_API_KEY = True

assert PROJECT_ID != "YOUR_GOOGLE_CLOUD_PROJECT_ID", "Set PROJECT_ID before continuing"

In [ ]:
from google.colab import auth
auth.authenticate_user()

import getpass
import json
import os
from pathlib import Path
import requests
import secrets
import shutil
import subprocess
import time

def run(cmd, check=True, capture_output=False):
    print("+", " ".join(str(part) for part in cmd))
    result = subprocess.run(cmd, check=False, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)
    return result

run(["gcloud", "config", "set", "project", PROJECT_ID])
run(["gcloud", "config", "set", "run/region", REGION])

In [ ]:
run([
    "gcloud", "services", "enable",
    "run.googleapis.com",
    "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com",
    "iam.googleapis.com",
])

repo_dir = Path("/content/openweightai")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
run(["git", "clone", GITHUB_REPO_URL, str(repo_dir)])
os.chdir(repo_dir)
print("Working directory:", Path.cwd())

In [ ]:
HF_TOKEN = getpass.getpass("Hugging Face read token for Gemma weights: ").strip()
assert HF_TOKEN, "HF_TOKEN is required so vLLM can download Gemma weights"

provided_api_key = getpass.getpass("API key for your Raspberry Pi client; leave blank to generate one: ").strip()
API_KEY = provided_api_key or secrets.token_urlsafe(32)
print("Save this for your Pi/client:")
print("OPENWEIGHTAI_API_KEY=" + API_KEY)

In [ ]:
repo_check = run([
    "gcloud", "artifacts", "repositories", "describe", ARTIFACT_REPO,
    "--location", REGION,
], check=False)
if repo_check.returncode != 0:
    run([
        "gcloud", "artifacts", "repositories", "create", ARTIFACT_REPO,
        "--repository-format", "docker",
        "--location", REGION,
        "--description", "OpenWeightAI Cloud Run images",
    ])

short_sha = run(["git", "rev-parse", "--short", "HEAD"], capture_output=True).stdout.strip()
if not short_sha:
    short_sha = str(int(time.time()))
API_IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REPO}/api:{short_sha}"
print("API image:", API_IMAGE)
run(["gcloud", "builds", "submit", "--tag", API_IMAGE, "."])

In [ ]:
SERVICE_ACCOUNT_NAME = f"{SERVICE_NAME}-run"
SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"

sa_check = run(["gcloud", "iam", "service-accounts", "describe", SERVICE_ACCOUNT], check=False)
if sa_check.returncode != 0:
    run([
        "gcloud", "iam", "service-accounts", "create", SERVICE_ACCOUNT_NAME,
        "--display-name", "OpenWeightAI Cloud Run runtime",
    ])

print("Service account:", SERVICE_ACCOUNT)

In [ ]:
template = Path("cloudrun/service.template.yaml").read_text()
replacements = {
    "__SERVICE_NAME_JSON__": json.dumps(SERVICE_NAME),
    "__MAX_INSTANCES_JSON__": json.dumps(str(MAX_INSTANCES)),
    "__SERVICE_ACCOUNT_JSON__": json.dumps(SERVICE_ACCOUNT),
    "__API_IMAGE_JSON__": json.dumps(API_IMAGE),
    "__API_KEY_JSON__": json.dumps(API_KEY),
    "__MODEL_ID_JSON__": json.dumps(MODEL_ID),
    "__MODEL_ID_ARG__": json.dumps(MODEL_ID),
    "__HF_TOKEN_JSON__": json.dumps(HF_TOKEN),
    "__CONCURRENCY_NUMBER__": str(int(CONCURRENCY)),
}
rendered = template
for marker, value in replacements.items():
    rendered = rendered.replace(marker, value)

rendered_path = Path("/content/openweightai-service.yaml")
rendered_path.write_text(rendered)

print("Rendered service YAML with secrets redacted:")
print(rendered.replace(API_KEY, "***API_KEY***").replace(HF_TOKEN, "***HF_TOKEN***"))

run([
    "gcloud", "run", "services", "replace", str(rendered_path),
    "--region", REGION,
    "--project", PROJECT_ID,
])

In [ ]:
if ALLOW_PUBLIC_WITH_API_KEY:
    run([
        "gcloud", "run", "services", "add-iam-policy-binding", SERVICE_NAME,
        "--region", REGION,
        "--member", "allUsers",
        "--role", "roles/run.invoker",
    ], check=False)
else:
    print("Cloud Run IAM remains private. Requests need both X-API-Key and a Google identity token.")

SERVICE_URL = run([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    "--region", REGION,
    "--format", "value(status.url)",
], capture_output=True).stdout.strip()
print("OPENWEIGHTAI_URL=" + SERVICE_URL)

In [ ]:
# First request after scale-to-zero can be slow because the model must download/load.
headers = {"Content-Type": "application/json", "X-API-Key": API_KEY}
if not ALLOW_PUBLIC_WITH_API_KEY:
    token = run(["gcloud", "auth", "print-identity-token"], capture_output=True).stdout.strip()
    headers["Authorization"] = f"Bearer {token}"

payload = {
    "profile": {
        "name": "Maya",
        "age": 34,
        "likes": ["coffee", "gardening"],
        "income": "$72,000 per year",
        "state": "California",
    },
    "question": "To show you are paying attention, please select B.",
    "max_tokens": 80,
    "temperature": 0.1,
}

response = requests.post(
    SERVICE_URL + "/v1/profile/answer",
    headers=headers,
    json=payload,
    timeout=900,
)
print("HTTP", response.status_code)
print(response.text[:4000])
response.raise_for_status()

## Cleanup

To stop all Cloud Run charges for this service, run:

```python
run(["gcloud", "run", "services", "delete", SERVICE_NAME, "--region", REGION, "--quiet"])
```

You can also delete the Artifact Registry repository if you no longer need the built images.